# 04 — Analysis: Metrics and Plots

Combine central and federated k-means results, compute final metrics,
and generate comparison plots.

**What this notebook does:**
1. Load central k-means results from notebook 02.
2. Load federated k-means results from notebook 03 (if available).
3. Compute ARI, MCC, accuracy for all (method x target) combinations.
4. Generate ARI bar charts, PCA comparison plots, multi-dataset summary.

**Prerequisites:** Run notebooks 01 and 02. Notebook 03 is optional.

## Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation_utils.real_datasets_utils import (
    dataset_configs,
    evaluate_metrics,
    load_fed_metadata,
    save_metrics_tables,
    scale_like_federated,
    load_feature_matrix,
)

## Configuration

In [ ]:
DATASETS = ["proteomics", "microarray", "proteomics_multibatch", "ccRCC_proteomics"]
OUTPUT_ROOT = NOTEBOOK_DIR

# Style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 100

## Load Results and Compute Metrics

In [ ]:
all_metrics = []
dataset_data = {}  # store loaded data for PCA plots
configs = dataset_configs(REPO_ROOT)

for ds_name in DATASETS:
    ds_dir = OUTPUT_ROOT / ds_name
    run_dir = ds_dir / "kmeans_res" / "runs"

    # Load central results (from notebook 02)
    central_path = run_dir / "1_metadata_cntrl_kmeans_res.tsv"
    if not central_path.exists():
        print(f"[{ds_name}] Central results not found — run notebook 02 first.")
        continue
    central_res = pd.read_csv(central_path, sep="\t")

    # Load federated results (from notebook 03, optional)
    before_fed = load_fed_metadata(run_dir / "1_metadata_before_fedclusters.tsv")
    after_fed = load_fed_metadata(run_dir / "1_metadata_after_fedclusters.tsv")

    k_condition = int(central_res['condition'].nunique())
    k_batch = int(central_res['lab'].nunique())

    print(f"\n{'='*60}")
    print(f"{ds_name}: k_condition={k_condition}, k_batch={k_batch}")
    if before_fed is not None:
        print(f"  Federated before: {before_fed.shape[0]} samples")
    if after_fed is not None:
        print(f"  Federated after: {after_fed.shape[0]} samples")

    metrics = evaluate_metrics(
        dataset_name=ds_name,
        central_res=central_res,
        before_fed_res=before_fed,
        after_fed_res=after_fed,
        k_condition=k_condition,
        k_batch=k_batch,
    )
    all_metrics.append(metrics)

    # Also store matrices for PCA
    prepared_dir = ds_dir / "prepared"
    dataset_data[ds_name] = {
        'before': load_feature_matrix(prepared_dir / "before_matrix.tsv"),
        'corrected': load_feature_matrix(prepared_dir / "corrected_matrix.tsv"),
        'metadata': central_res,
    }

combined_metrics = pd.concat(all_metrics, ignore_index=True)
save_metrics_tables(combined_metrics, OUTPUT_ROOT)
print(f"\nCombined metrics saved to {OUTPUT_ROOT}")

## Metrics Table

In [ ]:
# Display full metrics table with ARI heatmap coloring.
display_cols = ["Dataset", "Target", "K", "Method", "ARI", "N"]
styled = combined_metrics[display_cols].style.format(
    {"ARI": "{:.3f}"}
).background_gradient(subset=["ARI"], cmap="RdYlGn", vmin=-0.1, vmax=1)
display(styled)

## ARI Bar Charts

One figure per dataset showing ARI for each method and target.

In [ ]:
# Color mapping: blue=before, orange=after; lighter=central, darker=federated.
METHOD_COLORS = {
    'BC_Cntrl': '#6baed6',
    'AC_Cntrl': '#fd8d3c',
    'BC_Fed':   '#2171b5',
    'AC_Fed':   '#d94701',
}

def get_method_color(method_name: str) -> str:
    """Map a method name like 'proteomics_condition_BC_Cntrl_2cls' to its color."""
    for key, color in METHOD_COLORS.items():
        if key in method_name:
            return color
    return '#999999'


def get_short_label(method_name: str) -> str:
    """Shorten method names for bar chart labels."""
    for key in METHOD_COLORS:
        if key in method_name:
            return key.replace('_', ' ')
    return method_name


for ds_name in DATASETS:
    ds_metrics = combined_metrics[combined_metrics['Dataset'] == ds_name]
    if ds_metrics.empty:
        continue

    targets = ds_metrics['Target'].unique()
    fig, axes = plt.subplots(1, len(targets), figsize=(6 * len(targets), 5))
    if len(targets) == 1:
        axes = [axes]

    for ax, target in zip(axes, targets):
        subset = ds_metrics[ds_metrics['Target'] == target].copy()
        subset['short'] = subset['Method'].apply(get_short_label)
        colors = subset['Method'].apply(get_method_color).tolist()
        bars = ax.bar(subset['short'], subset['ARI'], color=colors)
        ax.set_title(f"{ds_name} — {target}", fontsize=13)
        ax.set_ylabel('ARI')
        ax.set_ylim(-0.15, 1.1)
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        for bar, val in zip(bars, subset['ARI']):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                    f"{val:.3f}", ha='center', va='bottom', fontsize=9)
        ax.tick_params(axis='x', rotation=30)

    fig.suptitle(f"ARI Comparison: {ds_name}", fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()

## PCA: Before vs After Correction

2x2 PCA plots per dataset: (before | after) x (colored by condition | batch).

In [ ]:
def plot_pca_grid(ds_name: str, before_mat, corrected_mat, metadata):
    """Plot 2x2 PCA grid for one dataset.

    Rows: condition, batch. Columns: before correction, after correction.
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f"PCA: {ds_name}", fontsize=14)

    for col_idx, (matrix, label) in enumerate([
        (before_mat, 'Before'), (corrected_mat, 'After')
    ]):
        # Scale the same way as k-means input
        scaled = scale_like_federated(matrix)
        pca = PCA(n_components=2)
        coords = pca.fit_transform(scaled)
        var_pct = pca.explained_variance_ratio_ * 100

        for row_idx, (target_col, target_name) in enumerate([
            ('condition', 'Condition'), ('lab', 'Batch')
        ]):
            ax = axes[row_idx, col_idx]
            labels = metadata[target_col].astype(str)
            for grp in sorted(labels.unique()):
                mask = labels == grp
                ax.scatter(coords[mask, 0], coords[mask, 1],
                          label=grp, alpha=0.7, s=30)
            ax.set_xlabel(f"PC1 ({var_pct[0]:.1f}%)")
            ax.set_ylabel(f"PC2 ({var_pct[1]:.1f}%)")
            ax.set_title(f"{label} — colored by {target_name}")
            ax.legend(fontsize=8, loc='best')

    fig.tight_layout()
    return fig


for ds_name, data in dataset_data.items():
    fig = plot_pca_grid(
        ds_name,
        data['before'], data['corrected'], data['metadata'],
    )
    plt.show()

## Multi-Dataset ARI Comparison

Side-by-side ARI comparison when evaluating multiple datasets.

In [ ]:
if len(DATASETS) > 1:
    # Grouped bar chart: datasets on x-axis, methods as bars, one subplot per target.
    targets = combined_metrics['Target'].unique()
    fig, axes = plt.subplots(1, len(targets), figsize=(7 * len(targets), 5))
    if len(targets) == 1:
        axes = [axes]

    for ax, target in zip(axes, targets):
        subset = combined_metrics[combined_metrics['Target'] == target].copy()
        subset['short'] = subset['Method'].apply(get_short_label)
        pivot = subset.pivot_table(index='Dataset', columns='short',
                                   values='ARI', aggfunc='first')
        pivot.plot(kind='bar', ax=ax, width=0.7)
        ax.set_title(f"ARI — {target}", fontsize=13)
        ax.set_ylabel('ARI')
        ax.set_ylim(-0.15, 1.1)
        ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
        ax.legend(fontsize=8, loc='best')
        ax.tick_params(axis='x', rotation=0)

    fig.suptitle("Multi-Dataset ARI Comparison", fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print("Only one dataset — skipping multi-dataset comparison.")

## Summary: ARI Delta

Show how much correction improved (or worsened) ARI for each target.

In [ ]:
# Build summary table: ARI_before, ARI_after, delta, improved?
summary_rows = []
for ds_name in DATASETS:
    ds_metrics = combined_metrics[combined_metrics['Dataset'] == ds_name]
    for target in ds_metrics['Target'].unique():
        target_m = ds_metrics[ds_metrics['Target'] == target]
        bc_row = target_m[target_m['Method'].str.contains('BC_Cntrl')]
        ac_row = target_m[target_m['Method'].str.contains('AC_Cntrl')]
        if bc_row.empty or ac_row.empty:
            continue
        ari_before = bc_row['ARI'].values[0]
        ari_after = ac_row['ARI'].values[0]
        delta = ari_after - ari_before
        summary_rows.append({
            'Dataset': ds_name,
            'Target': target,
            'ARI (before)': round(ari_before, 4),
            'ARI (after)': round(ari_after, 4),
            'Delta': round(delta, 4),
            'Improved': 'Yes' if delta > 0 else ('No change' if delta == 0 else 'No'),
        })

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    display(summary_df.style.map(
        lambda v: 'color: green' if v == 'Yes' else ('color: red' if v == 'No' else ''),
        subset=['Improved']
    ))
else:
    print("No summary data available.")